# NovaCart — Silver Order Payments Transformation

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Define Storage Paths

In [0]:
BRONZE_ORDER_PAYMENTS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/order_payments"
)

SILVER_ORDER_PAYMENTS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_payments"
)

QUARANTINE_ORDER_PAYMENTS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/order_payments"
)

print(f"Bronze path: {BRONZE_ORDER_PAYMENTS_PATH}")
print(f"Silver path: {SILVER_ORDER_PAYMENTS_PATH}")
print(f"Quarantine path: {QUARANTINE_ORDER_PAYMENTS_PATH}")

## 3. Read Bronze Order Payments Data

In [0]:
order_payments_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_ORDER_PAYMENTS_PATH)
)

bronze_row_count = order_payments_bronze_df.count()

print("Bronze order payments loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

order_payments_bronze_df.printSchema()
display(order_payments_bronze_df.limit(10))

## 4. Validate Required Columns

In [0]:
required_columns = [
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in order_payments_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 5. Profile Payment Types

In [0]:
display(
    order_payments_bronze_df
    .groupBy("payment_type")
    .count()
    .orderBy(F.desc("count"))
)

## 6. Profile Missing and Invalid Values

In [0]:
order_payments_profile_df = order_payments_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ).cast("int")
    ).alias("invalid_order_id"),

    F.sum(
        (
            F.col("payment_sequential").isNull()
            | (F.col("payment_sequential") <= 0)
        ).cast("int")
    ).alias("invalid_payment_sequential"),

    F.sum(
        (
            F.col("payment_type").isNull()
            | (F.trim(F.col("payment_type")) == "")
        ).cast("int")
    ).alias("invalid_payment_type"),

    F.sum(
        (
            F.col("payment_installments").isNull()
            | (F.col("payment_installments") < 0)
        ).cast("int")
    ).alias("invalid_payment_installments"),

    F.sum(
        F.col("payment_value").isNull().cast("int")
    ).alias("missing_payment_value"),

    F.sum(
        (F.col("payment_value") < 0).cast("int")
    ).alias("negative_payment_value"),

    F.sum(
        (F.col("payment_value") == 0).cast("int")
    ).alias("zero_payment_value"),
)

display(order_payments_profile_df)

## 7. Check Duplicate Payment Keys

In [0]:
duplicate_payment_keys_df = (
    order_payments_bronze_df
    .groupBy(
        "order_id",
        "payment_sequential"
    )
    .count()
    .filter(
        F.col("order_id").isNotNull()
        & F.col("payment_sequential").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_payment_key_count = duplicate_payment_keys_df.count()

print(
    "Number of duplicate "
    "(order_id, payment_sequential) keys: "
    f"{duplicate_payment_key_count}"
)

display(duplicate_payment_keys_df.limit(20))

## 8. Check Exact Duplicate Records

In [0]:
business_columns = [
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
]

exact_duplicate_count = (
    bronze_row_count
    - order_payments_bronze_df
        .dropDuplicates(business_columns)
        .count()
)

print(f"Exact duplicate order payment rows: {exact_duplicate_count}")

## 9. Inspect Zero-Value Payments

In [0]:
display(
    order_payments_bronze_df
    .filter(F.col("payment_value") == 0)
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
    .orderBy(
        "payment_type",
        "order_id",
        "payment_sequential"
    )
)

## 10. Profile Payment Installments

In [0]:
display(
    order_payments_bronze_df
    .groupBy(
        "payment_type",
        "payment_installments"
    )
    .count()
    .orderBy(
        "payment_type",
        "payment_installments"
    )
)

## 11. Confirm Duplicate Results

In [0]:
print(
    f"Duplicate payment keys: "
    f"{duplicate_payment_key_count}"
)

print(
    f"Exact duplicate payment rows: "
    f"{exact_duplicate_count}"
)

## 12. Clean and Standardize Payment Fields

In [0]:
order_payments_cleaned_df = (
    order_payments_bronze_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )
)

## 13. Define Allowed Payment Types

In [0]:
payment_key_window = Window.partitionBy(
    "order_id",
    "payment_sequential"
)

order_payments_checked_df = (
    order_payments_cleaned_df
    .withColumn(
        "_duplicate_key_count",
        F.count("*").over(payment_key_window)
    )
)
allowed_payment_types = [
    "credit_card",
    "boleto",
    "voucher",
    "debit_card",
]

## 14. Define Payment Validation Rules

In [0]:
invalid_order_id_condition = (
    F.col("order_id").isNull()
    | (F.col("order_id") == "")
)

invalid_payment_sequential_condition = (
    F.col("payment_sequential").isNull()
    | (F.col("payment_sequential") <= 0)
)

invalid_payment_type_condition = (
    F.col("payment_type").isNull()
    | (F.col("payment_type") == "")
    | ~F.col("payment_type").isin(allowed_payment_types)
)

invalid_payment_installments_condition = (
    F.col("payment_installments").isNull()
    | (F.col("payment_installments") < 0)
)

invalid_payment_value_condition = (
    F.col("payment_value").isNull()
    | (F.col("payment_value") < 0)
)

## 15. Assign Payment Rejection Reasons

In [0]:
order_payments_validated_df = order_payments_checked_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_order_id_condition,
        F.lit("MISSING_ORDER_ID")
    )
    .when(
        invalid_payment_sequential_condition,
        F.lit("INVALID_PAYMENT_SEQUENTIAL")
    )
    .when(
        invalid_payment_type_condition,
        F.lit("INVALID_PAYMENT_TYPE")
    )
    .when(
        invalid_payment_installments_condition,
        F.lit("INVALID_PAYMENT_INSTALLMENTS")
    )
    .when(
        invalid_payment_value_condition,
        F.lit("INVALID_PAYMENT_VALUE")
    )
    .when(
        F.col("_duplicate_key_count") > 1,
        F.lit("DUPLICATE_PAYMENT_KEY")
    )
    .otherwise(F.lit(None))
)

## 16. Review Payment Validation Results

In [0]:
display(
    order_payments_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 17. Split Valid and Invalid Payments

In [0]:
order_payments_valid_df = (
    order_payments_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop(
        "_rejection_reason",
        "_duplicate_key_count"
    )
)

order_payments_quarantine_df = (
    order_payments_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
    .drop("_duplicate_key_count")
)

## 18. Add Silver Processing Metadata

In [0]:
order_payments_silver_df = (
    order_payments_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 19. Add Quarantine Metadata

In [0]:
order_payments_quarantine_df = (
    order_payments_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("order_payments")
    )
)

## 20. Count Silver and Quarantine Records

In [0]:
valid_row_count = order_payments_silver_df.count()
quarantine_row_count = order_payments_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 21. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 22. Write Valid Payments to Silver

In [0]:
(
    order_payments_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_ORDER_PAYMENTS_PATH)
)

print("Silver order payments written successfully.")

## 23. Write Invalid Payments to Quarantine

In [0]:
(
    order_payments_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_ORDER_PAYMENTS_PATH)
)

print("Order payments quarantine output written successfully.")

## 24. Read Written Delta Outputs

In [0]:
order_payments_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_PAYMENTS_PATH)
)

order_payments_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_ORDER_PAYMENTS_PATH)
)

silver_written_count = order_payments_silver_written_df.count()
quarantine_written_count = order_payments_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 25. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver order payments pipeline completed successfully.")
print("Final row-count validation passed.")

## 26. Inspect Final Silver Payments Dataset

In [0]:
order_payments_silver_written_df.printSchema()

display(
    order_payments_silver_written_df.select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value",
        "_silver_processed_at"
    ).limit(20)
)